# 🚀 Notebook 3: Scaling Live Comments

A cooking stream with 50 viewers is easy. But what happens when the World Cup final goes live and **100 million people** are watching? Thousands of comments per second, millions of open connections, servers groaning under the load.

This notebook explores how to scale a live comments system from "works on my laptop" to "handles the Super Bowl."

## Learning Objectives

By the end of this notebook, you'll understand:
- Why a single server can't handle millions of viewers
- How pub/sub fan-out distributes comments across servers
- Channel partitioning to reduce wasted work
- Viewer co-location for efficient routing
- Mega-stream strategies: comment sampling and CDN snapshots

## ⚙️ Setup

Make sure the infrastructure is running before starting this notebook:

```bash
# From the fb-live-comments directory
docker compose up -d
```

**Services:**
- **PostgreSQL** — `localhost:5432` (user: `demo`, password: `demo`, db: `live_comments`)
- **Redis** — `localhost:6379`
- **Adminer** — [http://localhost:8080](http://localhost:8080) (Database UI)
- **RedisInsight** — [http://localhost:5540](http://localhost:5540) (Redis UI)

**Kernel:** Make sure you've selected the `.venv` kernel in the top-right of VS Code's kernel picker.
If it doesn't appear, reload the window (`Cmd+Shift+P` → "Reload Window").

```bash
# Create venv and install dependencies (if you haven't already)
uv venv
source .venv/bin/activate
uv sync
```

In [ ]:
import psycopg2
import redis

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="live_comments",
    user="demo",
    password="demo"
)
conn.autocommit = True
cur = conn.cursor()

# Connect to Redis
r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# Test PostgreSQL
cur.execute("SELECT COUNT(*) FROM comments")
comment_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM users")
user_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM live_videos")
video_count = cur.fetchone()[0]

# Test Redis
r.ping()

print("✅ Connected to PostgreSQL and Redis!")
print(f"   Database has {comment_count} comments, {user_count} users, {video_count} live videos")
print(f"   Redis is ready")

## 🖥️ The Single Server Limit

A single server can handle roughly **~100K concurrent SSE connections**. The limits come from:

- **File descriptors** — each open connection uses a file descriptor (OS limit)
- **Memory** — each connection needs buffers for sending data (~10-50KB each)
- **CPU** — serializing and sending comments to each connection takes cycles

With 100 million viewers, you'd need **1,000+ servers**. But when viewers are spread across
servers, how does a new comment reach ALL of them?

```
Server 1: [UserA, UserB, UserC] watching Video 1
Server 2: [UserD, UserE]        watching Video 1
Server 3: [UserF]               watching Video 1

Comment posted on Video 1 → hits Server 1
✅ Server 1 sends to UserA, UserB, UserC
❌ Server 2 and 3 don't know about the comment!
```

We need a way for servers to **coordinate** — that's where pub/sub comes in.

In [ ]:
import time, threading, json, redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# Simulate: how many SSE connections can we track per server?
# (We won't actually open connections, just simulate the data structures)

class SimulatedServer:
    """Simulates a Realtime Messaging Server."""

    def __init__(self, name, max_connections=100_000):
        self.name = name
        self.max_connections = max_connections
        # Map: video_id → set of connection IDs
        self.connections = {}
        self.total_connections = 0

    def connect(self, video_id, connection_id):
        if self.total_connections >= self.max_connections:
            return False
        if video_id not in self.connections:
            self.connections[video_id] = set()
        self.connections[video_id].add(connection_id)
        self.total_connections += 1
        return True

    def broadcast(self, video_id, comment):
        """Send comment to all viewers of this video on this server."""
        viewers = self.connections.get(video_id, set())
        return len(viewers)  # number of viewers reached

# Simulate: single server handling a popular stream
server = SimulatedServer("Server-1")

# 50,000 viewers connect to watch video 1
for i in range(50_000):
    server.connect(1, f"conn_{i}")

reached = server.broadcast(1, {"message": "Hello!"})
print(f"🖥️  Single Server Simulation")
print(f"   Connections: {server.total_connections:,}")
print(f"   Viewers reached: {reached:,}")
print(f"   Max capacity: {server.max_connections:,}")
print(f"   Utilization: {server.total_connections / server.max_connections * 100:.0f}%")
print()
print(f"⚠️  But we need 100M viewers... that's {100_000_000 // server.max_connections:,} servers!")
print(f"   And each server only knows about ITS viewers.")
print(f"   We need a way to coordinate between servers.")

## 📢 Solution 1: Simple Pub/Sub (Broadcast Everything)

The simplest approach: all servers subscribe to a **single Redis pub/sub channel**.
When a comment is posted, it's published to that channel. Every server receives every
comment and checks locally if any of its viewers care about it.

```
Comment posted → PUBLISH to "all_comments" channel
                         ↓
Server 1 (subscribes) ← receives ALL comments → filters for its viewers
Server 2 (subscribes) ← receives ALL comments → filters for its viewers
Server 3 (subscribes) ← receives ALL comments → filters for its viewers
```

**The problem?** If there are 100,000 live videos, Server 1 might only care about
3 of them — but it still receives (and discards) comments for the other 99,997.
That's a LOT of wasted bandwidth and CPU.

In [ ]:
# === One shared workload for all three strategies ===
# The original version of this comparison gave each strategy a DIFFERENT set of
# viewers, so "relevant messages" differed between them and the table was
# meaningless. Fix the workload once; vary only the routing.

import random

NUM_VIDEOS   = 100
NUM_SERVERS  = 10
NUM_COMMENTS = 1000

random.seed(123)
COMMENTS = [{"video_id": random.randint(1, NUM_VIDEOS), "message": f"Comment {i}"}
            for i in range(NUM_COMMENTS)]

# Every video has viewers somewhere. Under the first two strategies the load
# balancer is video-agnostic, so a video's viewers land on an arbitrary server.
random.seed(42)
_videos = list(range(1, NUM_VIDEOS + 1))
random.shuffle(_videos)
RANDOM_ASSIGNMENT = {                        # server_index -> set of video ids
    i: set(_videos[i::NUM_SERVERS]) for i in range(NUM_SERVERS)
}

print("Shared workload")
print("=" * 60)
print(f"  {NUM_VIDEOS} live videos, {NUM_COMMENTS} comments, {NUM_SERVERS} realtime servers")
print(f"  Videos per server (random routing): "
      f"{[len(v) for v in RANDOM_ASSIGNMENT.values()]}")
print(f"  Every comment is relevant to exactly one server, so a perfect system")
print(f"  would deliver exactly {NUM_COMMENTS:,} messages.")

In [ ]:
class PubSubServer:
    """Server that subscribes to one global channel and filters locally."""

    def __init__(self, name, video_ids):
        self.name = name
        self.video_ids = video_ids
        self.received = 0
        self.relevant = 0

    def process_comment(self, comment):
        self.received += 1
        if comment["video_id"] in self.video_ids:
            self.relevant += 1


servers = [PubSubServer(f"Server-{i+1}", RANDOM_ASSIGNMENT[i]) for i in range(NUM_SERVERS)]

# Every server hears every comment.
for comment in COMMENTS:
    for server in servers:
        server.process_comment(comment)

total_received = sum(s.received for s in servers)
total_relevant = sum(s.relevant for s in servers)
waste_pct = (1 - total_relevant / total_received) * 100

print("📢 Simple Pub/Sub — Broadcasting Everything")
print("=" * 60)
for s in servers:
    s_waste = (1 - s.relevant / s.received) * 100
    print(f"   {s.name}: received {s.received:>5}, relevant {s.relevant:>4}, wasted {s_waste:.0f}%")

print(f"\n📊 Total messages delivered: {total_received:,}")
print(f"   Actually relevant:        {total_relevant:,}")
print(f"   Wasted:                   {waste_pct:.0f}%")
print(f"   Amplification:            {total_received / total_relevant:.1f}x the useful work")
print(f"\n⚠️  Amplification here is exactly NUM_SERVERS ({NUM_SERVERS}x). Add servers to")
print("   handle more viewers and every server gets proportionally busier —")
print("   this design does not scale horizontally at all.")

## 🔀 Solution 2: Channel Partitioning

Instead of one giant channel, create **N channels** (partitions). Each video maps to a
channel via `hash(video_id) % N`. Servers only subscribe to channels that contain videos
their viewers are watching.

```
Video 1  → hash(1) % 10  = Channel 1
Video 2  → hash(2) % 10  = Channel 2
Video 15 → hash(15) % 10 = Channel 5

Server A (viewers for Video 1, Video 15) → subscribes to Channel 1, Channel 5
Server B (viewers for Video 2)           → subscribes to Channel 2
```

Now each server only receives comments for videos that **hash to the same channel**
as the videos it cares about.

Here's the part that usually gets glossed over: **this barely helps on its own.**
If a server's videos are scattered randomly across all N channels, it subscribes
to nearly all N channels and is right back to receiving everything. Partitioning
only pays when a server's videos are *clustered* into a few channels. Watch the
numbers below — then notice that co-location is precisely what creates that
clustering.

In [ ]:
NUM_CHANNELS = 10

def get_channel(video_id):
    """Map a video ID to a pub/sub channel."""
    return f"comments_channel:{video_id % NUM_CHANNELS}"


class PartitionedServer:
    """Server that subscribes only to the channels its videos hash into."""

    def __init__(self, name, video_ids):
        self.name = name
        self.video_ids = video_ids
        self.subscribed_channels = {get_channel(vid) for vid in video_ids}
        self.received = 0
        self.relevant = 0

    def process_comment(self, comment):
        self.received += 1
        if comment["video_id"] in self.video_ids:
            self.relevant += 1


servers_partitioned = [PartitionedServer(f"Server-{i+1}", RANDOM_ASSIGNMENT[i])
                       for i in range(NUM_SERVERS)]

for comment in COMMENTS:
    channel = get_channel(comment["video_id"])
    for server in servers_partitioned:
        if channel in server.subscribed_channels:
            server.process_comment(comment)

total_received_p = sum(s.received for s in servers_partitioned)
total_relevant_p = sum(s.relevant for s in servers_partitioned)
waste_pct_p = (1 - total_relevant_p / total_received_p) * 100

print("🔀 Channel Partitioning — Subscribe to Relevant Channels Only")
print("=" * 60)
print(f"   {NUM_CHANNELS} channels\n")
for s in servers_partitioned:
    s_waste = (1 - s.relevant / s.received) * 100 if s.received else 0
    print(f"   {s.name}: channels {len(s.subscribed_channels):>2}, "
          f"received {s.received:>5}, relevant {s.relevant:>4}, wasted {s_waste:.0f}%")

print(f"\n📊 Total messages delivered: {total_received_p:,}")
print(f"   Actually relevant:        {total_relevant_p:,}")
print(f"   Wasted:                   {waste_pct_p:.0f}%")
print(f"   Amplification:            {total_received_p / total_relevant_p:.1f}x")
print()
print(f"😐 Barely better than broadcasting ({waste_pct:.0f}% → {waste_pct_p:.0f}% waste). Why?")
print(f"   Each server holds ~{NUM_VIDEOS // NUM_SERVERS} videos scattered at random across")
print(f"   {NUM_CHANNELS} channels, so it ends up subscribed to almost all of them.")
print("   Partitioning only pays off when a server's videos CLUSTER into a few")
print("   channels — which is exactly what co-location arranges next.")

## 🎯 Solution 3: Viewer Co-location

The ultimate optimization — route viewers of the **same video to the same server**.

How? Use **consistent hashing** on `video_id` in the load balancer (Layer 7).
When a viewer connects to watch Video 42, the load balancer hashes `42` and always
routes to the same server. This means:

- Each server only handles viewers for specific videos
- Each server subscribes to exactly the channels it needs
- Every message it receives is relevant — zero waste!

### The Improvement Progression

```
Simple Pub/Sub     → Every server gets every comment      → 10x amplification
Channel Partition  → Servers get comments for N channels  → ~7x amplification
Viewer Co-location → Servers get only relevant comments   → 1x (zero waste)
```

Run the cells and check these against the printed output — the partitioning win
is much smaller than most write-ups admit, and the reason is worth understanding.

In [ ]:
import hashlib

def consistent_hash(video_id, num_servers):
    """Consistent hash: maps video_id to a server index."""
    h = hashlib.md5(str(video_id).encode()).hexdigest()
    return int(h, 16) % num_servers


class ColocatedServer:
    def __init__(self, name):
        self.name = name
        self.video_ids = set()
        self.received = 0
        self.relevant = 0

    def process_comment(self, comment):
        self.received += 1
        if comment["video_id"] in self.video_ids:
            self.relevant += 1


colocated_servers = [ColocatedServer(f"Server-{i+1}") for i in range(NUM_SERVERS)]

# The load balancer hashes video_id, so all viewers of a video land together.
for video_id in range(1, NUM_VIDEOS + 1):
    colocated_servers[consistent_hash(video_id, NUM_SERVERS)].video_ids.add(video_id)

# Same comments, routed by the same hash.
for comment in COMMENTS:
    colocated_servers[consistent_hash(comment["video_id"], NUM_SERVERS)].process_comment(comment)

total_received_c = sum(s.received for s in colocated_servers)
total_relevant_c = sum(s.relevant for s in colocated_servers)
waste_pct_c = (1 - total_relevant_c / total_received_c) * 100

print("🎯 Viewer Co-location — Route Viewers to the Right Server")
print("=" * 60)
for s in colocated_servers:
    pct = (1 - s.relevant / s.received) * 100 if s.received else 0
    print(f"   {s.name}: videos={len(s.video_ids):<3} received={s.received:<5} "
          f"relevant={s.relevant:<5} waste={pct:.0f}%")

print(f"\n📊 Total messages delivered: {total_received_c:,}")
print(f"   Actually relevant:        {total_relevant_c:,}")
print(f"   Wasted:                   {waste_pct_c:.0f}%")
print(f"   Amplification:            {total_received_c / total_relevant_c:.1f}x")
print()
print("✅ Zero waste. But look at the per-server video counts — the hash does NOT")
print("   spread evenly, and that is before you account for videos having wildly")
print("   different viewer counts. One World Cup final hashes to ONE server and")
print("   melts it. Co-location trades fan-out waste for hot-shard risk, which is")
print("   what the rest of this notebook is about.")

## 📊 Scaling Strategy Comparison

Let's compare all three strategies side by side. The key metric is **efficiency** —
what percentage of delivered messages were actually relevant to the receiving server?

| Strategy | How it Works | Amplification | What it costs you |
|----------|-------------|---------------|-------------------|
| Simple Pub/Sub | Every server gets every comment | ×(number of servers) | Grows *linearly as you add servers* — the opposite of scaling |
| Channel Partitioning | Servers subscribe to hashed channels | ×(channels subscribed ÷ channels needed) | Only helps if a server's videos cluster into a few channels |
| Viewer Co-location | Layer-7 LB hashes `video_id` | ×1 | Sticky routing, painful rebalancing, and **hot shards** — one viral video lands on exactly one server |

In [ ]:
strategies = [
    ("Simple Pub/Sub",     total_received,   total_relevant),
    ("Channel Partition",  total_received_p, total_relevant_p),
    ("Viewer Co-location", total_received_c, total_relevant_c),
]

print("📊 Message Delivery Efficiency — same workload, same useful work")
print("=" * 78)
print(f"{'Strategy':<22} {'Delivered':>11} {'Relevant':>10} {'Efficiency':>12} {'Amplification':>15}")
print("-" * 78)

for name, delivered, relevant in strategies:
    eff = relevant / delivered * 100
    amp = delivered / relevant
    bar = "█" * int(eff / 5) + "░" * (20 - int(eff / 5))
    print(f"{name:<22} {delivered:>11,} {relevant:>10,} {eff:>11.0f}% {amp:>14.1f}x  {bar}")

print()
print("Every row does the same 1,000 units of useful work. The only thing that")
print("changes is how much pointless work each server does alongside it.")

## 🌋 The Mega-Stream Problem

Everything above works great when viewers are spread across many videos. But what
about **ONE video** with **100 million viewers**?

Think: World Cup final, a presidential debate, a surprise celebrity livestream.

The math gets scary:
- 100M viewers ÷ 100K per server = **1,000 servers** just for that ONE video
- 5,000 comments/second → every server must broadcast ALL of them to its viewers
- At 5,000 comments/sec, each comment is visible for **4 milliseconds**. Nobody can read that fast.

### Key Insight

When comments flow this fast, the **goal changes**. Users aren't reading a conversation
— they're feeling the **"energy of the crowd."** They want to see:
- The general mood (🎉🎉🎉 after a goal)
- The occasional standout comment
- That the stream is "alive" and active

This changes our engineering approach entirely. We don't need to deliver **every**
comment to **every** viewer. We need to deliver a **representative sample** that
captures the energy.

## 🎲 Mega-Stream Strategy 1: Comment Sampling

Instead of delivering every comment, **sample a subset**. Adjust the sampling rate
based on how fast comments are coming in (the "velocity"):

With a target of **30 comments/sec** per viewer, the sample rate is simply
`target ÷ velocity`:

| Comment Velocity | Sample Rate = 30 ÷ velocity | Viewer Sees |
|-----------------|-------------|-------------|
| 20 comments/sec | 100% (already under target) | 20 comments/sec |
| 100 comments/sec | 30% | ~30 comments/sec |
| 1,000 comments/sec | 3% | ~30 comments/sec |
| 5,000 comments/sec | 0.6% | ~30 comments/sec |

Each viewer sees **at most 30 comments/sec** regardless of actual volume — a
comfortable, readable pace.

**What sampling costs:** it is lossy *by design*. A viewer will never see most
comments, so "did my comment even post?" becomes a real support question. The
standard patch is **read-your-own-writes** — always deliver a user their own
comment unsampled, even when nobody else sees it.

**Prioritization** (optional): Instead of random sampling, prioritize:
1. Comments from users you follow
2. Verified / creator accounts
3. Comments with reactions
4. Random fill for the rest

In [ ]:
import random

class CommentSampler:
    """Drops comments so each viewer sees at most `target_rate` per second."""

    def __init__(self, target_rate=30):
        self.target_rate = target_rate
        self.current_rate = 0        # measured incoming comments/sec

    def get_sample_rate(self):
        """Fraction of comments to show. Never above 1.0."""
        if self.current_rate <= self.target_rate:
            return 1.0
        return self.target_rate / self.current_rate

    def should_show(self, comment):
        return random.random() < self.get_sample_rate()


print("🎲 Comment Sampling at Different Velocities (target 30/s per viewer)")
print("=" * 74)
print(f"  {'incoming/s':>11}  {'sample rate':>12}  {'viewer sees/s':>14}  {'expected':>9}")

random.seed(7)  # deterministic output; the point is the ratio, not the noise
for velocity in [20, 100, 1000, 5000]:
    sampler = CommentSampler(target_rate=30)
    sampler.current_rate = velocity

    # Average over several simulated seconds so the sample isn't noisy.
    seconds = 20
    shown = sum(1 for _ in range(velocity * seconds) if sampler.should_show({})) / seconds
    expected = velocity * sampler.get_sample_rate()

    print(f"  {velocity:>11,}  {sampler.get_sample_rate():>11.1%}  "
          f"{shown:>14.1f}  {expected:>9.1f}")
    assert abs(shown - expected) < max(3, expected * 0.15), "sampler is off target"

print()
print("💡 The incoming rate spans 250x; what the viewer sees does not move.")
print("   They still feel the 'energy of the crowd' without being overwhelmed.")
print("   Cost: at 5,000/s a viewer sees 0.6% of comments. Deliver a user their")
print("   OWN comment unsampled, or they will think posting is broken.")

## 🌐 Mega-Stream Strategy 2: CDN Snapshots

This is perhaps the most elegant solution for truly massive streams.

Instead of **pushing** comments via SSE (one connection per viewer), switch to a
**pull** model:

1. Server maintains a **ring buffer** of recent 100-200 comments
2. Every ~1 second, snapshot this buffer and push to **CDN**
3. Clients **poll the CDN** every second
4. Client animates comments smoothly between polls

```
Ring Buffer (server) → snapshot every 1s → CDN Edge Cache
                                              ↑
                            Millions of clients poll CDN
```

**Why this works so well:**
- CDNs are designed to serve the SAME content to millions of users
- No per-connection state on the server
- No pub/sub, no SSE connections, no WebSockets
- The CDN does all the heavy lifting

**Trade-off:** Latency goes from ~200ms (SSE) to ~1-2s (polling). But for a
mega-stream where comments fly by in milliseconds anyway, nobody notices.

In [ ]:
from collections import deque
import json, time

class RingBuffer:
    """Fixed-size buffer of recent comments, like a circular queue."""

    def __init__(self, max_size=200):
        self.buffer = deque(maxlen=max_size)

    def add(self, comment):
        self.buffer.append(comment)

    def snapshot(self):
        """Return current state as a JSON-serializable list."""
        return list(self.buffer)

class CDNSimulator:
    """Simulates a CDN edge cache."""

    def __init__(self):
        self.cached_snapshot = None
        self.cache_time = None

    def update(self, snapshot):
        """Origin pushes a new snapshot."""
        self.cached_snapshot = json.dumps(snapshot)
        self.cache_time = time.time()

    def fetch(self):
        """Client fetches from CDN (very fast, edge-cached)."""
        if self.cached_snapshot:
            return json.loads(self.cached_snapshot)
        return []

# Simulate a mega-stream
buffer = RingBuffer(max_size=100)
cdn = CDNSimulator()

# 5000 comments arrive in 1 second
for i in range(5000):
    buffer.add({
        "id": i + 1,
        "user": f"User{random.randint(1, 1000000)}",
        "message": f"Comment #{i+1} 🎉"
    })

# Take snapshot and push to CDN
snapshot = buffer.snapshot()
cdn.update(snapshot)

# Simulate 10K clients fetching from CDN
start = time.time()
for _ in range(10000):
    comments = cdn.fetch()
elapsed = (time.time() - start) * 1000

print("🌐 CDN Snapshot Simulation")
print("=" * 55)
print(f"   Comments in ring buffer: {len(buffer.buffer)}")
print(f"   Snapshot size: {len(cdn.cached_snapshot):,} bytes")
print(f"   10,000 CDN fetches: {elapsed:.0f}ms ({elapsed/10000:.3f}ms each)")
print(f"   Latest comment in snapshot: #{snapshot[-1]['id']}")
print()
print("💡 Key insight: CDN serves the SAME snapshot to millions of clients.")
print("   No per-connection state, no pub/sub, no SSE connections.")
print("   The CDN is doing what it was literally designed to do!")
print()
print(f"📊 Comparison for 10M viewers, 5000 comments/sec:")
print(f"   SSE approach:  10M open connections, 5000 × 10M = 50B messages/sec")
print(f"   CDN approach:  10M polls/sec to CDN (trivial for CDN), ~1s latency")

## 🔄 When to Use Each Strategy

Here's a decision tree for choosing the right scaling strategy:

```
Viewers < 100K?
  → Standard SSE + Redis Pub/Sub (Notebook 1)

100K < Viewers < 10M?
  → Channel partitioning + viewer co-location
  → Comment sampling if comments/sec > 500

Viewers > 10M (Mega-stream)?
  → CDN snapshots + client-side animation
  → SSE only for the commenter's "read your own write"
```

| Strategy | Viewers | Latency | Complexity | When |
|----------|---------|---------|------------|------|
| Simple Pub/Sub | < 100K | ~200ms | Low | Most live videos |
| Channel Partitioning | < 1M | ~200ms | Medium | Popular videos |
| Viewer Co-location | < 10M | ~200ms | High | Very popular videos |
| Comment Sampling | Any | ~200ms | Medium | High comment velocity |
| CDN Snapshots | Unlimited | ~1-2s | Medium | Mega-viral events |

In [ ]:
import redis, json, threading, time

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

NUM_CHANNELS = 5

def get_channel(video_id):
    return f"live_comments_ch:{video_id % NUM_CHANNELS}"

# Subscriber: simulate a server subscribing to specific channels
received_messages = []

def subscriber_worker(channel_name, stop_event):
    sub_r = redis.Redis(host="localhost", port=6379, decode_responses=True)
    pubsub = sub_r.pubsub()
    pubsub.subscribe(channel_name)
    while not stop_event.is_set():
        msg = pubsub.get_message(timeout=0.5)
        if msg and msg["type"] == "message":
            received_messages.append(json.loads(msg["data"]))
    pubsub.unsubscribe()
    sub_r.close()

# Subscribe to channel for video 1
channel = get_channel(1)
stop_event = threading.Event()
sub_thread = threading.Thread(target=subscriber_worker, args=(channel, stop_event), daemon=True)
sub_thread.start()
time.sleep(0.5)

# Publish comments for different videos
for video_id in [1, 2, 3, 1, 4, 1]:
    ch = get_channel(video_id)
    comment = {"video_id": video_id, "message": f"Comment for video {video_id}"}
    r.publish(ch, json.dumps(comment))
    time.sleep(0.1)

time.sleep(1)
stop_event.set()
sub_thread.join(timeout=2)

print(f"🔀 Channel Partitioning with Redis Pub/Sub")
print(f"   Subscribed to: {channel}")
print(f"   Comments received: {len(received_messages)}")
for msg in received_messages:
    relevant = "✅" if msg["video_id"] == 1 else "⚠️  (same channel, different video)"
    print(f"   {relevant} Video {msg['video_id']}: {msg['message']}")

In [ ]:
# Clean up Redis keys
r = redis.Redis(host="localhost", port=6379, decode_responses=True)
for key in r.keys("live_comments_ch:*"):
    r.delete(key)
print("🧹 Cleaned up Redis keys")

## ✍️ Don't Forget the Write Path

So far we've focused on **reading** (delivering comments to viewers). But the **write path** — the POST that creates a comment — has its own scaling problems on a mega-stream.

Imagine 100 million viewers, and **5,000 people hit "post" every second**. Every comment means:

1. A database `INSERT` (durable storage)
2. A Redis `PUBLISH` (real-time fan-out)
3. Possibly moderation checks (profanity, spam)

If the database does **5,000 INSERTs/sec for a single video**, one `comments` table becomes a bottleneck. Here are the real-world tricks used to survive:

### 1. Rate limiting per user
Nobody needs to post more than ~1 comment per second. A token-bucket limiter in front of the API rejects the firehose from bots and button-mashers **before** it touches your DB.

### 2. Async write queue
Instead of writing to Postgres synchronously on every POST, push the comment onto a queue (Kafka, Redis Stream, SQS) and return `202 Accepted` immediately. Workers drain the queue and batch-insert.

```
POST /comments  →  validate  →  enqueue  →  PUBLISH to Redis  →  202 Accepted
                                    ↓
                       (async worker batches 100 rows → 1 INSERT)
```

The viewer sees the comment instantly via pub/sub. The DB sees one bulk insert instead of thousands of single-row inserts.

### 3. Moderation pipeline
At scale, you can't let every comment go live without filtering. Common layers:
- **Cheap checks inline** (length, rate limit, block-list keywords)
- **Async ML checks** (toxicity, spam) — if flagged, retract via a "delete" pub/sub event
- **Human review queue** for edge cases

### 4. Shard hot videos
If a single viral video gets 10K writes/sec, a single `comments` row sequence becomes a lock hotspot. Shard writes across partitions keyed by `(video_id, hash(comment_id))` and merge on read.

Let's simulate the difference between synchronous and batched writes.


In [ ]:
# Measure the write path for real against the Postgres in docker-compose.
# The original version of this cell multiplied two invented constants together,
# which teaches nothing. Actual inserts, actual timings.

import time
import psycopg2.extras

NUM_COMMENTS = 2000
BATCH_SIZE = 100
VIDEO = 2

wconn = psycopg2.connect(host="localhost", port=5432, dbname="live_comments",
                         user="demo", password="demo")
wconn.autocommit = True   # commit per statement, like a real request handler
wcur = wconn.cursor()

rows = [(VIDEO, 1, f"writepath-bench {i}") for i in range(NUM_COMMENTS)]

# ── 1. Synchronous: one INSERT + one COMMIT per comment ─────────────────
t0 = time.time()
for row in rows:
    wcur.execute(
        "INSERT INTO comments (live_video_id, user_id, message) VALUES (%s, %s, %s)", row
    )
sync_ms = (time.time() - t0) * 1000

wcur.execute("DELETE FROM comments WHERE message LIKE 'writepath-bench%'")

# ── 2. Batched: one multi-row INSERT per BATCH_SIZE comments ────────────
t0 = time.time()
for start in range(0, NUM_COMMENTS, BATCH_SIZE):
    psycopg2.extras.execute_values(
        wcur,
        "INSERT INTO comments (live_video_id, user_id, message) VALUES %s",
        rows[start:start + BATCH_SIZE],
    )
batched_ms = (time.time() - t0) * 1000

wcur.execute("SELECT COUNT(*) FROM comments WHERE message LIKE 'writepath-bench%'")
inserted = wcur.fetchone()[0]
wcur.execute("DELETE FROM comments WHERE message LIKE 'writepath-bench%'")
wconn.close()

assert inserted == NUM_COMMENTS, f"batched insert lost rows: {inserted}"

print(f"✍️  Write Path: {NUM_COMMENTS:,} comments into Postgres")
print("=" * 62)
label_sync = "Synchronous (1 INSERT + 1 COMMIT each)"
label_batch = f"Batched ({BATCH_SIZE} rows per INSERT)"
print(f"   {label_sync:<40} {sync_ms:>8,.0f} ms  "
      f"({NUM_COMMENTS / (sync_ms / 1000):>8,.0f} rows/s)")
print(f"   {label_batch:<40} {batched_ms:>8,.0f} ms  "
      f"({NUM_COMMENTS / (batched_ms / 1000):>8,.0f} rows/s)")
print(f"   Speedup: {sync_ms / batched_ms:.1f}x")
print()
print("💡 What batching actually costs you:")
print(f"   • Up to {BATCH_SIZE} comments sit in memory before they are durable. If the")
print("     worker crashes, they are gone — unless the queue in front of it is")
print("     durable (Kafka/SQS), which is the whole point of putting one there.")
print("   • Added write latency: a comment waits for its batch to fill or for the")
print("     flush timer. Viewers do not notice, because Redis Pub/Sub already")
print("     delivered it — but 'posted' and 'durable' are now different moments.")
print("   • Bigger batches keep helping until you hit lock and WAL contention;")
print("     100–1000 rows is the usual sweet spot. Measure, do not guess.")

## 🌍 How Real Systems Do It

The strategies in this notebook aren't hypothetical — they're how the big platforms actually work:

| Platform | Strategy You've Seen | Real-World Note |
|----------|---------------------|-----------------|
| **Facebook Live** | Pub/Sub + viewer co-location | Uses MQTT over persistent TCP to mobile devices; the original inspiration for this lab |
| **YouTube Live Chat** | CDN snapshots + polling | Chat polls every ~5s on most streams; you'll see the `get_live_chat` endpoint in the API |
| **Twitch** | IRC-derived WebSocket + sharded chat servers | Channels sharded by channel name; TMI (Twitch Messaging Interface) fronts thousands of IRC-style backends |
| **Instagram Live** | Comment sampling at extreme scale | Only a fraction of comments are delivered to each viewer during viral streams |
| **Twitter/X Spaces** | WebRTC for audio + WS for reactions | Reactions are aggressively sampled; you rarely see every 🎉 |
| **Discord** | WebSocket gateway + Cassandra | Per-channel fan-out via gateway servers; messages stored in Cassandra partitioned by `channel_id` |

### What They All Share

1. **Separate read/write paths** — commenters hit one service, viewers hit another
2. **A pub/sub layer in the middle** — Kafka, Redis, or a custom gossip protocol
3. **Sampling or rate limiting on the hot path** — never deliver more than humans can read
4. **Graceful degradation under load** — drop features (reactions, typing indicators) before dropping comments

### A Good Interview Answer

> "I'd start with SSE + Redis Pub/Sub for the basic case. As the stream gets hot, I'd add a Layer-7 load balancer with consistent hashing on `video_id` so viewers of the same video co-locate on the same realtime server — that eliminates pub/sub fan-out waste. On the write side, I'd put a rate limiter in front and use an async write queue to batch DB inserts. For mega-streams — think Super Bowl scale — I'd flip from SSE push to CDN-cached snapshots with ~1s polling, because a CDN can serve identical snapshots to millions of clients essentially for free."


## 📚 Summary

### Key Takeaways

1. **Single servers have limits** — ~100K connections max; horizontal scaling is required
2. **Simple pub/sub wastes resources** — every server gets every comment regardless of relevance
3. **Channel partitioning** reduces waste — servers subscribe only to relevant channels
4. **Viewer co-location** eliminates waste — route viewers to the server that handles their video
5. **Mega-streams need different thinking** — at extreme scale, sample comments or switch to CDN pull

### The Scaling Ladder

```
Small stream     →  Single server + Redis Pub/Sub
Popular stream   →  Multiple servers + channel partitioning
Viral stream     →  Viewer co-location + consistent hashing
Mega-stream      →  Comment sampling + CDN snapshots
```

### How This Maps to the System Design Interview

| Concept | What to Say |
|---------|-------------|
| Horizontal scaling | "Separate read (SSE) and write (POST) servers; scale independently" |
| Pub/Sub coordination | "Redis Pub/Sub broadcasts comments; channel partitioning reduces fan-out" |
| Viewer co-location | "Layer 7 load balancer with consistent hashing routes viewers of same video to same server" |
| Mega-streams | "At extreme scale, switch from push to pull: CDN snapshots with 1s polling" |
| Comment sampling | "When comments exceed human reading speed, sample a representative subset" |

### 🎓 You've Completed the Lab!

You now understand the full architecture of a live comments system — from a single-server
prototype to a design that handles hundreds of millions of viewers. These concepts apply
to any real-time system: live chat, collaborative editing, stock tickers, gaming, and more.